## 1. Imports ##

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR

import duckdb
import polars as pl

## 2. DuckDB Connection ##

In [4]:
con = duckdb.connect()

con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='8GB'")

PARQUET = PROCESSED_DATA_DIR / "players_clean"

## 3. Dataset Overview ##

In [5]:
con.sql(f"""
SELECT

COUNT(*) AS total_players,

COUNT(DISTINCT club_name) AS clubs,

COUNT(DISTINCT league_name) AS leagues,

COUNT(DISTINCT nationality_name) AS nationalities

FROM parquet_scan('{PARQUET}')
""")

┌───────────────┬───────┬─────────┬───────────────┐
│ total_players │ clubs │ leagues │ nationalities │
│     int64     │ int64 │  int64  │     int64     │
├───────────────┼───────┼─────────┼───────────────┤
│      10003590 │  1648 │      44 │           193 │
└───────────────┴───────┴─────────┴───────────────┘

## 4. Missing Value Summary ##

In [6]:
con.sql(f"""
SELECT

SUM(value_eur IS NULL) AS missing_value,

SUM(wage_eur IS NULL) AS missing_wage,

SUM(age IS NULL) AS missing_age,

SUM(overall IS NULL) AS missing_overall

FROM parquet_scan('{PARQUET}')
""")

┌───────────────┬──────────────┬─────────────┬─────────────────┐
│ missing_value │ missing_wage │ missing_age │ missing_overall │
│    int128     │    int128    │   int128    │     int128      │
├───────────────┼──────────────┼─────────────┼─────────────────┤
│        132797 │       115974 │           0 │               0 │
└───────────────┴──────────────┴─────────────┴─────────────────┘

## 5. Feature Engineering ##

In [9]:
FEATURES = con.sql(f"""
SELECT

player_id,

short_name,

club_name,

league_name,

nationality_name,

CAST(overall AS INTEGER) AS overall,

CAST(potential AS INTEGER) AS potential,

CAST(age AS INTEGER) AS age,

CAST(value_eur AS DOUBLE) AS value_eur,

CAST(wage_eur AS DOUBLE) AS wage_eur,

CAST(pace AS DOUBLE) AS pace,

CAST(shooting AS DOUBLE) AS shooting,

CAST(passing AS DOUBLE) AS passing,

CAST(dribbling AS DOUBLE) AS dribbling,

CAST(defending AS DOUBLE) AS defending,

CAST(physic AS DOUBLE) AS physic,

CAST(potential AS INTEGER)
-
CAST(overall AS INTEGER)
AS improvement,

CAST(value_eur AS DOUBLE)
/
NULLIF(CAST(wage_eur AS DOUBLE),0)
AS value_per_salary,

(
CAST(power_strength AS DOUBLE)
+
CAST(power_stamina AS DOUBLE)
)/2
AS physical_score,

(
CAST(attacking_finishing AS DOUBLE)
+
CAST(shooting AS DOUBLE)
)/2
AS attacking_score,

(
CAST(defending AS DOUBLE)
+
CAST(mentality_interceptions AS DOUBLE)
)/2
AS defending_score

FROM parquet_scan('{PARQUET}')
""").df()



## 6. Inspect ##

In [10]:
FEATURES.head()

,player_id,short_name,club_name,league_name,nationality_name,overall,potential,age,value_eur,wage_eur,...,shooting,passing,dribbling,defending,physic,improvement,value_per_salary,physical_score,attacking_score,defending_score
0,158023,L. Messi,Paris Saint Germain,Ligue 1,Argentina,91,91,35,54000000.0,195000.0,...,89.0,90.0,94.0,34.0,64.0,0,276.923077,69.0,89.5,37.0
1,165153,K. Benzema,Real Madrid,La Liga,France,91,91,34,64000000.0,450000.0,...,88.0,83.0,87.0,39.0,78.0,0,142.222222,82.0,90.0,39.0
2,188545,R. Lewandowski,FC Barcelona,La Liga,Poland,91,91,33,84000000.0,420000.0,...,91.0,79.0,86.0,44.0,83.0,0,200.000000,81.5,92.5,46.5
3,192985,K. De Bruyne,Manchester City,Premier League,Belgium,91,91,31,107500000.0,350000.0,...,88.0,93.0,87.0,63.0,77.0,0,307.142857,81.5,86.5,63.5
4,231747,K. Mbappé,Paris Saint Germain,Ligue 1,France,91,95,23,190500000.0,230000.0,...,89.0,80.0,92.0,36.0,76.0,4,828.260870,81.5,91.0,37.0


## 7. Null Handling ##

In [12]:
numeric_cols = FEATURES.select_dtypes(include="number").columns

FEATURES[numeric_cols] = FEATURES[numeric_cols].fillna(
    FEATURES[numeric_cols].mean()
)

## 8. Derived Categories ##

In [13]:
import numpy as np

FEATURES["career_stage"] = np.select(
    [
        FEATURES["age"] < 21,
        FEATURES["age"] < 30,
    ],
    [
        "Young",
        "Prime",
    ],
    default="Veteran",
)

## 9. Percentile Ranking ##

In [14]:
FEATURES["overall_rank"] = (
    FEATURES["overall"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

## 10. Top Players ##

In [16]:
FEATURES.sort_values(
    "overall",
    ascending=False
).head(20)

,player_id,short_name,club_name,league_name,nationality_name,overall,potential,age,value_eur,wage_eur,...,dribbling,defending,physic,improvement,value_per_salary,physical_score,attacking_score,defending_score,career_stage,overall_rank
4486789,158023,L. Messi,FC Barcelona,La Liga,Argentina,94,94,31,110500000.0,575000.0,...,96.0,32.0,61.0,0,192.173913,65.5,93.0,27.0,Veteran,1
2582753,158023,L. Messi,FC Barcelona,La Liga,Argentina,94,94,32,95500000.0,560000.0,...,96.0,39.0,66.0,0,170.535714,71.5,93.5,39.5,Veteran,1
3358111,158023,L. Messi,FC Barcelona,La Liga,Argentina,94,94,32,95500000.0,560000.0,...,96.0,39.0,66.0,0,170.535714,71.5,93.5,39.5,Veteran,1
6345753,20801,Cristiano Ronaldo,Real Madrid,La Liga,Portugal,94,94,32,95500000.0,575000.0,...,90.0,33.0,80.0,0,166.086957,86.0,93.5,31.0,Veteran,1
7017248,20801,Cristiano Ronaldo,Real Madrid,La Liga,Portugal,94,94,31,87000000.0,575000.0,...,91.0,33.0,80.0,0,151.304348,86.0,92.5,31.0,Veteran,1
5175110,158023,L. Messi,FC Barcelona,La Liga,Argentina,94,94,30,118500000.0,575000.0,...,96.0,26.0,61.0,0,206.086957,66.0,92.5,24.0,Veteran,1
5175109,20801,Cristiano Ronaldo,Real Madrid,La Liga,Portugal,94,94,32,95500000.0,575000.0,...,90.0,33.0,80.0,0,166.086957,86.0,93.5,31.0,Veteran,1
7631252,20801,Cristiano Ronaldo,Real Madrid,La Liga,Portugal,94,94,31,87000000.0,575000.0,...,91.0,33.0,80.0,0,151.304348,86.0,92.5,31.0,Veteran,1
5588498,20801,Cristiano Ronaldo,Real Madrid,La Liga,Portugal,94,94,32,95500000.0,575000.0,...,90.0,33.0,80.0,0,166.086957,86.0,93.5,31.0,Veteran,1
4432658,158023,L. Messi,FC Barcelona,La Liga,Argentina,94,94,31,110500000.0,575000.0,...,96.0,32.0,61.0,0,192.173913,65.5,93.0,27.0,Veteran,1


## 11. League Statistics ##

In [18]:
league_stats = con.sql(f"""
SELECT

league_name,

AVG(TRY_CAST(overall AS DOUBLE)) AS avg_overall,

AVG(TRY_CAST(value_eur AS DOUBLE)) AS avg_value,

AVG(TRY_CAST(age AS DOUBLE)) AS avg_age,

COUNT(*) AS players

FROM parquet_scan('{PARQUET}')

GROUP BY league_name

ORDER BY avg_overall DESC
""").df()

## 12. Club Statistics ##

In [20]:
club_stats = con.sql(f"""
SELECT

club_name,

AVG(TRY_CAST(overall AS DOUBLE)) AS avg_overall,

AVG(TRY_CAST(value_eur AS DOUBLE)) AS avg_value,

COUNT(*) AS squad_size

FROM parquet_scan('{PARQUET}')

GROUP BY club_name
""").df()

## 13. Nationality Statistics ##

In [21]:
country_stats = con.sql(f"""
SELECT

nationality_name,

COUNT(*) AS players,

AVG(TRY_CAST(overall AS DOUBLE)) AS avg_rating

FROM parquet_scan('{PARQUET}')

GROUP BY nationality_name
""").df()

## 14. Save Engineering Features ##

In [22]:
FEATURES.to_parquet(
    PROCESSED_DATA_DIR / "features.parquet",
    index=False,
)

league_stats.to_parquet(
    PROCESSED_DATA_DIR / "league_stats.parquet",
    index=False,
)

club_stats.to_parquet(
    PROCESSED_DATA_DIR / "club_stats.parquet",
    index=False,
)

country_stats.to_parquet(
    PROCESSED_DATA_DIR / "country_stats.parquet",
    index=False,
)